In [1]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind

df = pd.read_csv("vanguard_master_cleaned.csv")

## High-balance Completion rate

In [2]:
client_summary = df.groupby("client_id").agg(
    Variation=("Variation", "first"),
    is_high_balance=("is_high_balance", "first"),
    completed=("process_step", lambda x: (x == "confirm").any()),
    clnt_tenure_yr=("clnt_tenure_yr", "first"),
    clnt_age=("clnt_age", "first")
).reset_index()

client_summary.head()

,client_id,Variation,is_high_balance,completed,clnt_tenure_yr,clnt_age
0,555,Test,False,True,3.0,29.5
1,647,Test,False,True,12.0,57.5
2,934,Test,False,False,9.0,51.0
3,1028,Control,False,False,12.0,36.0
4,1104,Control,True,False,5.0,48.0


In [3]:
high_balance_clients = client_summary[
    client_summary["is_high_balance"] == True
]

hb_completion = high_balance_clients.groupby("Variation").agg(
    total_clients=("client_id", "nunique"),
    completed_clients=("completed", "sum")
)

hb_completion["completion_rate"] = hb_completion["completed_clients"] / hb_completion["total_clients"]

hb_completion

,total_clients,completed_clients,completion_rate
Variation,,,
Control,5427,3623,0.667588
Test,6177,4355,0.705035


In [5]:
count = hb_completion.loc[["Test", "Control"], "completed_clients"]
nobs = hb_completion.loc[["Test", "Control"], "total_clients"]

z_stat, p_value = proportions_ztest(
    count=count,
    nobs=nobs,
    alternative="two-sided"
)

z_stat, p_value

(np.float64(4.342367795357343), np.float64(1.4095530576390116e-05))

In [6]:
import pandas as pd

df = pd.read_csv("vanguard_master_cleaned.csv")

In [7]:
df.head()
df.columns

Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time',
       'Variation', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'clnt_age', 'gendr',
       'num_accts', 'bal', 'calls_6_mnth', 'logons_6_mnth', 'is_high_balance',
       'is_multi_account'],
      dtype='object')

In [8]:
step_summary = df.groupby("client_id").agg(
    Variation=("Variation", "first"),
    is_high_balance=("is_high_balance", "first"),
    reached_step_1=("process_step", lambda x: (x == "step_1").any()),
    reached_step_2=("process_step", lambda x: (x == "step_2").any()),
    reached_step_3=("process_step", lambda x: (x == "step_3").any()),
    completed=("process_step", lambda x: (x == "confirm").any())
).reset_index()

In [9]:
from statsmodels.stats.proportion import proportions_ztest

for step in ["reached_step_1", "reached_step_2", "reached_step_3", "completed"]:

    temp = step_summary[
        step_summary["is_high_balance"]
    ].groupby("Variation").agg(
        total=("client_id", "nunique"),
        success=(step, "sum")
    )

    count = temp["success"]
    nobs = temp["total"]

    z, p = proportions_ztest(count, nobs)

    print(step)
    print(temp)
    print("p-value:", p)
    print()

reached_step_1
           total  success
Variation                
Control     5427     4673
Test        6177     5518
p-value: 1.1555829007570561e-07

reached_step_2
           total  success
Variation                
Control     5427     4285
Test        6177     5103
p-value: 5.760547585311451e-07

reached_step_3
           total  success
Variation                
Control     5427     4059
Test        6177     4856
p-value: 1.1272677034676181e-06

completed
           total  success
Variation                
Control     5427     3623
Test        6177     4355
p-value: 1.4095530576390116e-05



For high-balance clients, the Test group had a completion rate of **70.50%**, compared with **66.76%** in the Control group. The two-proportion Z-test produced a p-value below 0.05, so we reject the null hypothesis. This means the difference is statistically significant. The new UI did not create hidden performance issues for high-balance clients; instead, it improved completion for this valuable segment.

## Additional Demographic Hipothesis

In [6]:
test_tenure = client_summary[
    client_summary["Variation"] == "Test"
]["clnt_tenure_yr"].dropna()

control_tenure = client_summary[
    client_summary["Variation"] == "Control"
]["clnt_tenure_yr"].dropna()

test_tenure.mean(), control_tenure.mean()

(np.float64(11.98290122769927), np.float64(12.087686487865007))

In [7]:
t_stat, p_value = ttest_ind(
    test_tenure,
    control_tenure,
    equal_var=False
)

t_stat, p_value

(np.float64(-1.7115258395121253), np.float64(0.08699034968593344))

The tenure t-test produced a p-value above 0.05, so we fail to reject the null hypothesis. This suggests there is no statistically significant difference in average client tenure between the Test and Control groups. This supports the idea that the experiment groups were reasonably balanced by tenure.

## Evaluation

# Duration Assessment

The experiment ran from March 15, 2017 to June 20, 2017, which is approximately three months. This timeframe appears sufficient for collecting a large sample size and observing user behavior across both Test and Control groups.

The dataset contains more than 50,000 unique clients and over 300,000 process events after cleaning, so the experiment had enough volume for statistical testing. The significant results from the completion-rate tests also suggest that the sample size was large enough to detect meaningful differences between groups.

However, a longer experiment could still be useful to confirm whether the results remain stable over time. Since financial behavior can vary by month, market conditions, or client urgency, extending the experiment could help confirm that the improvement is consistent and not only related to a specific time period.

Overall, the three-month duration was sufficient for initial A/B test evaluation, but Vanguard should continue monitoring the redesigned UI before a full rollout.

## Additional Data Needs

To improve the personalization prompt strategy, Vanguard would benefit from collecting additional user-level and session-level data.

Useful additional data could include:

- Device type: desktop, mobile, or tablet
- Browser type
- New vs returning visitor status
- Whether the client used help, FAQ, or support during the process
- Clickstream details before abandonment
- Reason for abandonment, if collected through a short exit survey
- Client investment goal or account type
- Time of day and day of week
- Whether the client had previous failed attempts

Device type would be especially important because users on mobile may experience different friction compared with desktop users. If high-balance clients abandon more often on mobile, Vanguard could create mobile-specific prompts or simplify certain steps.

Help interaction data would also be useful. If users open help pages or pause for a long time at a specific step, Vanguard could trigger an in-context prompt at that moment.

Overall, additional behavioral and technical data would help Vanguard personalize prompts more accurately instead of showing the same generic message to every client.

# Conclusion

The experiment duration appears sufficient for the initial analysis because the sample size was large and the statistical tests produced clear results. However, Vanguard should continue monitoring performance after launch to confirm that the improvement remains stable over time.

For future personalization, Vanguard should collect more context about device type, user behavior, help usage, and abandonment reasons. This would allow prompts to be targeted to the right users at the right step, especially high-balance clients who may need additional support early in the process.